# Text Embeddings: model statyczny a enkoder kontekstowy

Zadanie 1, część II. Porównujemy dwa sposoby osadzania zdań z korpusu PolEmo (`all.sentence.train.txt`):

- **model statyczny:** Word2Vec IPI PAN (formy, 300 wymiarów), wektor zdania to średnia wektorów znanych słów,
- **enkoder kontekstowy:** `sdadas/mmlw-roberta-base` (SentenceTransformers).

> Etykiety w tym notebooku to oryginalne etykiety wydźwięku zdań z korpusu PolEmo, a nie nasza anotacja z Label Studio.

## 1. Importy i konfiguracja

In [18]:
import re
import textwrap
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from gensim.models import KeyedVectors
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split

In [ ]:
SEED = 42
N_SAMPLE = 5000

DATA = Path("../data/dataset_conll/all.sentence.train.txt")

W2V_NAME = "nkjp+wiki-forms-all-300-skipg-ns"
W2V_URL = f"https://dsmodels.nlp.ipipan.waw.pl/dsmodels/{W2V_NAME}.txt.gz"
W2V_PATH = Path(f"../data/models/{W2V_NAME}.txt.gz")
W2V_LIMIT = 1_000_000

ST_MODEL = "sdadas/mmlw-roberta-base"

## 2. Dane

### 2.1 Wczytanie korpusu

Format fastText: jedna linia to jedno zdanie, a `__label__...` na końcu to etykieta. Tekst jest już stokenizowany spacjami. Do enkodera kontekstowego odtwarzamy czytelną formę (`text`), a Word2Vec dostaje tokeny z `raw`.

In [ ]:
def detokenize(text: str) -> str:
    text = re.sub(r" ([.,!?;:)%])", r"\1", text)
    return re.sub(r"\( ", "(", text)


def parse(line: str) -> dict:
    raw, label = line.rsplit(" __label__", 1)
    raw = raw.strip()
    return {
        "raw": raw,
        "text": detokenize(raw),
        "label": label.strip().removeprefix("z_"),
    }


lines = [l for l in DATA.read_text(encoding="utf-8").splitlines() if l.strip()]
df = pd.DataFrame([parse(l) for l in lines])
df["n_tokens"] = df["raw"].str.split().str.len()

print(f"{len(df):,} zdań")
df["label"].value_counts()

45,974 zdań


label
minus_m    16911
plus_m     12293
zero       11255
amb         5515
Name: count, dtype: int64

### 2.2 Próbka stratyfikowana

t-SNE i interaktywny wykres z 46 tys. punktów byłyby bardzo wolne, więc bierzemy losową próbkę z zachowaniem proporcji klas.

In [ ]:
sample, _ = train_test_split(
    df, train_size=N_SAMPLE, stratify=df["label"], random_state=SEED
)
sample = sample.reset_index(drop=True)
sample["tokens"] = sample["raw"].str.lower().str.split()

sample["label"].value_counts(normalize=True).round(3)

label
minus_m    0.368
plus_m     0.267
zero       0.245
amb        0.120
Name: proportion, dtype: float64

## 3. Osadzenia

### 3.1 Word2Vec (formy)

Model IPI PAN trenowany na formach słów (NKJP + Wikipedia), skip-gram z negative sampling. Plik waży ok. 2,3 GB i trafia do `../data/models/`.

In [5]:
W2V_PATH.parent.mkdir(parents=True, exist_ok=True)
if not W2V_PATH.exists():
    urllib.request.urlretrieve(W2V_URL, W2V_PATH)

w2v = KeyedVectors.load_word2vec_format(W2V_PATH, limit=W2V_LIMIT)
print(f"Słownik: {len(w2v):,} form, wymiar {w2v.vector_size}")

Słownik: 1,000,000 form, wymiar 300


Wektor zdania to średnia wektorów znanych tokenów. Tokeny spoza słownika (np. słowa bez polskich znaków) są pomijane, a zdania bez żadnego znanego tokenu wykluczamy z próbki, żeby oba modele porównywać na tych samych zdaniach.

In [6]:
def sentence_vector(tokens: list[str]) -> np.ndarray | None:
    vectors = [w2v[t] for t in tokens if t in w2v]
    return np.mean(vectors, axis=0) if vectors else None


all_tokens = [t for tokens in sample["tokens"] for t in tokens]
print(f"Pokrycie słownika: {np.mean([t in w2v for t in all_tokens]):.1%} tokenów")

vectors = [sentence_vector(tokens) for tokens in sample["tokens"]]
keep = np.array([v is not None for v in vectors])
print(f"Wykluczone zdania bez znanego tokenu: {(~keep).sum()}")

sample = sample[keep].reset_index(drop=True)
emb_w2v = np.stack([v for v in vectors if v is not None])
emb_w2v.shape

Pokrycie słownika: 82.0% tokenów
Wykluczone zdania bez znanego tokenu: 8


(4992, 300)

### 3.2 mmlw-roberta-base

Karta modelu wymaga prefiksu `zapytanie: ` tylko dla zapytań w zadaniach wyszukiwania. Nasze zdania traktujemy jak zwykłe teksty, więc nie dodajemy prefiksu. Wektory mają 768 wymiarów.

In [7]:
st_model = SentenceTransformer(ST_MODEL)
emb_st = st_model.encode(sample["text"].tolist(), batch_size=64, show_progress_bar=True)
emb_st.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/78 [00:00<?, ?it/s]

(4992, 768)

In [8]:
embeddings = {"Word2Vec (formy)": emb_w2v, "mmlw-roberta-base": emb_st}

## 4. Wizualizacja t-SNE

### 4.1 Redukcja wymiarowości

Ta sama metoda i parametry dla obu modeli, żeby różnice wynikały z osadzeń, a nie z redukcji. t-SNE nie zachowuje odległości globalnych, więc rozmiaru klastrów i dystansów między nimi nie interpretujemy.

In [ ]:
def tsne_2d(X: np.ndarray) -> np.ndarray:
    tsne = TSNE(
        n_components=2, perplexity=30, metric="cosine", init="pca", random_state=SEED
    )
    return tsne.fit_transform(X)


coords = {name: tsne_2d(X) for name, X in embeddings.items()}

### 4.2 Wykresy interaktywne

Kolor to etykieta wydźwięku, a po najechaniu na punkt widać zdanie, etykietę i liczbę tokenów.

In [ ]:
sample["hover"] = sample["text"].map(lambda t: "<br>".join(textwrap.wrap(t, 70)[:6]))

plot_df = pd.concat(
    [sample.assign(model=name, x=xy[:, 0], y=xy[:, 1]) for name, xy in coords.items()]
)

fig = px.scatter(
    plot_df,
    x="x",
    y="y",
    color="label",
    facet_col="model",
    hover_name="hover",
    hover_data={
        "label": True,
        "n_tokens": True,
        "x": False,
        "y": False,
        "model": False,
        "hover": False,
    },
    category_orders={"label": ["plus_m", "minus_m", "zero", "amb"]},
    opacity=0.7,
    height=600,
)
fig.update_traces(marker_size=5)
fig.update_xaxes(matches=None, showticklabels=False, title=None)
fig.update_yaxes(matches=None, showticklabels=False, title=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

### 4.3 Miara separacji klas

Silhouette liczymy w oryginalnej przestrzeni osadzeń (odległość kosinusowa), a nie po redukcji, bo t-SNE zniekształca odległości. Wartość bliska 0 oznacza nakładające się klasy, a wyższa lepszą separację.

In [ ]:
silhouette = pd.Series(
    {
        name: silhouette_score(X, sample["label"], metric="cosine")
        for name, X in embeddings.items()
    },
    name="silhouette (kosinus)",
)
silhouette.round(3).to_frame()

,silhouette (kosinus)
Word2Vec (formy),-0.038
mmlw-roberta-base,0.013


Sylwetka bliska zeru, co mówi, ze klasy sa bardzo nakladajace się.

## 5. Analiza osadzeń

Wszystkie pary szukamy automatycznie na całej próbce. Do podobieństwa używamy cosinusa, a pary identycznych tekstów pomijamy, bo inaczej wygrywałyby powtórzenia o cosinusie 1,0.

In [12]:
def unit(X: np.ndarray) -> np.ndarray:
    return X / np.linalg.norm(X, axis=1, keepdims=True)


labels = sample["label"].to_numpy()
texts = sample["text"].to_numpy()
text_codes = pd.factorize(sample["text"])[0]

# górny trójkąt macierzy podobieństw: każda para tylko raz
pair_i, pair_j = np.triu_indices(len(sample), k=1)
is_duplicate = text_codes[pair_i] == text_codes[pair_j]
different_label = labels[pair_i] != labels[pair_j]

cosine = {name: (unit(X) @ unit(X).T)[pair_i, pair_j] for name, X in embeddings.items()}
static_name, contextual_name = embeddings

print(f"{len(pair_i):,} par, w tym {is_duplicate.sum()} identycznych tekstów")

12,457,536 par, w tym 1 identycznych tekstów


In [ ]:
def pairs_table(
    scores: np.ndarray, mask: np.ndarray, n: int = 10, lowest: bool = False
) -> pd.DataFrame:
    candidates = np.flatnonzero(mask)
    order = candidates[np.argsort(scores[candidates])]
    top = order[:n] if lowest else order[::-1][:n]
    i, j = pair_i[top], pair_j[top]
    return pd.DataFrame(
        {
            "cosinus": scores[top].round(3),
            "etykieta_1": labels[i],
            "tekst_1": [t[:150] for t in texts[i]],
            "etykieta_2": labels[j],
            "tekst_2": [t[:150] for t in texts[j]],
        }
    )

### 5.1 Podobne wektory, różne etykiety

Dziesięć par o najwyższym cosinusie, ale różnych etykietach, osobno dla każdego modelu. Takie pary pokazują, co model uznaje za bliskie mimo odmiennego wydźwięku (np. przez negację).

In [14]:
pd.concat(
    {
        name: pairs_table(s, different_label & ~is_duplicate)
        for name, s in cosine.items()
    },
    names=["model"],
)

cosinus etykieta_1  \
model                                     
Word2Vec (formy)  0    1.000     plus_m   
                  1    1.000     plus_m   
                  2    0.951     plus_m   
                  3    0.944        amb   
                  4    0.942       zero   
                  5    0.941    minus_m   
                  6    0.941    minus_m   
                  7    0.941       zero   
                  8    0.940    minus_m   
                  9    0.940    minus_m   
mmlw-roberta-base 0    0.993        amb   
                  1    0.982     plus_m   
                  2    0.982     plus_m   
                  3    0.980     plus_m   
                  4    0.979     plus_m   
                  5    0.978     plus_m   
                  6    0.977        amb   
                  7    0.975     plus_m   
                  8    0.974       zero   
                  9    0.973    minus_m   

                                                               tekst_1  \
model                                                                    
Word2Vec (formy)  0                                         Polecam!!!   
                  1                                    Polecam Marcin.   
                  2                               Z aneksem kuchennym.   
                  3  Pokój dostali śmy tak okropny ze musiałem dopł...   
                  4  Na drugi dzień poszła m do innego lekarza i do...   
                  5  A teraz szczegóły: - przez cztery dni pobytu m...   
                  6  szkoda słów krytyki na tego typu lekarza.... m...   
                  7  Na drugi dzień poszła m do innego lekarza i do...   
                  8  Tymczasem miała m kilka niepokojących objawów,...   
                  9  Zdaje sobie sprawę z tego, że mój przypadek by...   
mmlw-roberta-base 0                                 Był em zszokowany.   
                  1             Hotel w pełni zasługuje na 4 gwiazdki.   
                  2                                Generalnie polecam!   
                  3                                           POLECAMY   
                  4     Śniadania są ok duży wybór na szwedzkim stole.   
                  5                                    Raczej polecam!   
                  6       Śniadania smaczne choć wybór raczej skromny.   
                  7                              Hotel nowy i zadbany.   
                  8                                     Miłego pobytu!   
                  9                                    RAZ JESZCZE!!!!   

                    etykieta_2  \
model                            
Word2Vec (formy)  0    minus_m   
                  1    minus_m   
                  2       zero   
                  3    minus_m   
                  4     plus_m   
                  5        amb   
                  6     plus_m   
                  7    minus_m   
                  8     plus_m   
                  9     plus_m   
mmlw-roberta-base 0    minus_m   
                  1        amb   
                  2        amb   
                  3    minus_m   
                  4       zero   
                  5    minus_m   
                  6    minus_m   
                  7    minus_m   
                  8     plus_m   
                  9     plus_m   

                                                               tekst_2  
model                                                                   
Word2Vec (formy)  0                                        Polecam....  
                  1                                        Polecam....  
                  2                        Pokoje z aneksem kuchennym.  
                  3  Jeden z niewielu plusów to dokładnie i codzien...  
                  4  Po swoim pobycie w szpitalu przeczytała m wiel...  
                  5  PLUSY: bardzo dobre jedzenie (w żadnym z europ...  
                  6  Po swoim pobycie w szpitalu przeczytała m wiel...  
                  7  Gdyby nie to że sa

### 5.2 Ta sama etykieta, odległe wektory

Dziesięć par o tej samej etykiecie i najniższym cosinusie. To słaby dowód na podobieństwo znaczenia, bo dwa zdania o tym samym wydźwięku mogą dotyczyć zupełnie różnych tematów (np. hotelu i lekarza), więc niski cosinus bywa poprawny.

In [15]:
pd.concat(
    {
        name: pairs_table(s, ~different_label & ~is_duplicate, lowest=True)
        for name, s in cosine.items()
    },
    names=["model"],
)

cosinus etykieta_1  \
model                                     
Word2Vec (formy)  0   -0.033       zero   
                  1   -0.025       zero   
                  2   -0.011    minus_m   
                  3   -0.008    minus_m   
                  4   -0.003       zero   
                  5   -0.002       zero   
                  6    0.002    minus_m   
                  7    0.005       zero   
                  8    0.007       zero   
                  9    0.009       zero   
mmlw-roberta-base 0    0.642       zero   
                  1    0.643       zero   
                  2    0.644       zero   
                  3    0.644       zero   
                  4    0.647       zero   
                  5    0.647       zero   
                  6    0.647       zero   
                  7    0.648       zero   
                  8    0.648       zero   
                  9    0.649       zero   

                                                               tekst_1  \
model                                                                    
Word2Vec (formy)  0                            Borkowska Burnecka lub.   
                  1                                            I było.   
                  2                                              Była.   
                  3                                             zadne.   
                  4                            Borkowska Burnecka lub.   
                  5  Laboratorium prowadzone zgodnie z planem Domar...   
                  6                                              Była.   
                  7  Czy usg i markery, które między innymi wchodzą...   
                  8                                         Bezczelna?   
                  9                                             Pokój?   
mmlw-roberta-base 0  Położone w stanie New Jersey Atlantic City jes...   
                  1  Żeby narodziny tej myszy były możliwe, naukowc...   
                  2  Bez spotkań naukowców nie ma rozwoju nauki, be...   
                  3  Co trzeci gość nie jest uczestnikiem żadnego h...   
                  4  Budowa Hampton by Hilton Warsaw Airport zlokal...   
                  5  Podstawowym badaniem pomocnym w ocenie czynnoś...   
                  6  Napaleni super fotkami z bookingu zdecydowali ...   
                  7  Co trzeci gość nie jest uczestnikiem żadnego h...   
                  8  Podstawowym badaniem pomocnym w ocenie czynnoś...   
                  9  Należy pamiętać, że przecież najważniejsza w p...   

                    etykieta_2  \
model                            
Word2Vec (formy)  0       zero   
                  1       zero   
                  2    minus_m   
                  3    minus_m   
                  4       zero   
                  5       zero   
                  6    minus_m   
                  7       zero   
                  8       zero   
                  9       zero   
mmlw-roberta-base 0       zero   
                  1       zero   
                  2       zero   
                  3       zero   
                  4       zero   
                  5       zero   
                  6       zero   
                  7       zero   
                  8       zero   
                  9       zero   

                                                               tekst_2  
model                                                                   
Word2Vec (formy)  0                                 Moje drogie Panie.  
                  1                                         Bezczelna?  
                  2                                    Drinki okropne.  
                  3                                              Była.  
                  4                                         Bezczelna?  
                  5                                         Bezczelna?  
                  6                                        Jakis zart.  
                  7                    

#### Mocniejszy dowód: modele się nie zgadzają

Korpus nie ma etykiet parafraz, więc szukamy par, które `mmlw-roberta-base` (trenowany na podobieństwo semantyczne) uznaje za bliskie, a Word2Vec za odległe, i które mają mało wspólnych słów. Wtedy różnica wynika ze słownictwa, a nie z treści.

Skale cosinusa w obu modelach różnią się (średnie wektory Word2Vec są do siebie bardziej podobne), więc porównujemy rangi percentylowe, a nie surowe wartości. Nakładanie słów mierzymy współczynnikiem Jaccarda zbiorów tokenów alfanumerycznych (bez interpunkcji).

In [19]:
N_CANDIDATES = 2000
MAX_JACCARD = 0.2

percentile = {
    name: pd.Series(s).rank(pct=True).to_numpy() for name, s in cosine.items()
}
gap = percentile[contextual_name] - percentile[static_name]

word_sets = [{t for t in tokens if t.isalnum()} for tokens in sample["tokens"]]


def jaccard(i: int, j: int) -> float:
    a, b = word_sets[i], word_sets[j]
    return len(a & b) / len(a | b) if a | b else 0.0


candidates = np.flatnonzero(~is_duplicate)
candidates = candidates[np.argsort(gap[candidates])[::-1][:N_CANDIDATES]]

disagreement = pd.DataFrame(
    {
        "jaccard": [jaccard(pair_i[k], pair_j[k]) for k in candidates],
        f"cosinus {static_name}": cosine[static_name][candidates],
        f"cosinus {contextual_name}": cosine[contextual_name][candidates],
        "etykieta_1": labels[pair_i[candidates]],
        "tekst_1": [t[:150] for t in texts[pair_i[candidates]]],
        "etykieta_2": labels[pair_j[candidates]],
        "tekst_2": [t[:150] for t in texts[pair_j[candidates]]],
    }
)
low_overlap = disagreement[disagreement["jaccard"] < MAX_JACCARD]
print(f"{len(low_overlap)} z {N_CANDIDATES} kandydatów ma Jaccard < {MAX_JACCARD}")
low_overlap.head(10).round(3)

2000 z 2000 kandydatów ma Jaccard < 0.2


,jaccard,cosinus Word2Vec (formy),cosinus mmlw-roberta-base,etykieta_1,tekst_1,etykieta_2,tekst_2
0,0.0,0.010,0.973,minus_m,RAZ JESZCZE!!!!,plus_m,POLECAMY
1,0.0,0.027,0.957,zero,Bezczelna?,plus_m,POLECAMY
2,0.0,0.069,0.971,minus_m,RAZ JESZCZE!!!!,minus_m,ODRADZAM!!!!
3,0.0,0.039,0.947,minus_m,Koniec!,minus_m,ODRADZAM!!!!
4,0.0,0.005,0.944,plus_m,Calkiem niezla.,zero,I było.
5,0.0,0.045,0.942,minus_m,RAZ JESZCZE!!!!,minus_m,Kpina!!!
6,0.0,0.078,0.959,zero,Bezczelna?,minus_m,PODSUMOWUJĄC... OMIJAJCIE SZEROKIM ŁUKIEM!
7,0.0,0.061,0.944,plus_m,Wow!!,plus_m,POLECAMY
8,0.0,0.078,0.946,plus_m,Calkiem niezla.,amb,Ale się zdążyło.
9,0.0,0.059,0.936,minus_m,tylko sobie!,plus_m,POLECAMY


### 5.3 Długość zdania a wektor

Dla obu modeli liczymy korelację rangową Spearmana `n_tokens` z:

- normą wektora (przy uśrednianiu wektorów słów norma zwykle maleje z długością),
- podobieństwem do centroidu własnej klasy (czy długie zdania leżą bliżej środka klasy).

Jeśli norma wektora jest stała (np. enkoder zwraca wektory jednostkowe), korelacja jest nieokreślona, co widać w kolumnie `std normy`.

In [20]:
def centroid_similarity(X: np.ndarray) -> np.ndarray:
    Xu = unit(X)
    centroids = {
        l: unit(Xu[labels == l].mean(axis=0, keepdims=True))[0]
        for l in np.unique(labels)
    }
    return (Xu * np.stack([centroids[l] for l in labels])).sum(axis=1)


n_tokens = sample["n_tokens"].to_numpy()
rows = {}
for name, X in embeddings.items():
    norm = np.linalg.norm(X, axis=1)
    rho_norm, p_norm = spearmanr(n_tokens, norm)
    rho_centroid, p_centroid = spearmanr(n_tokens, centroid_similarity(X))
    rows[name] = {
        "std normy": norm.std(),
        "rho (norma)": rho_norm,
        "p (norma)": p_norm,
        "rho (centroid)": rho_centroid,
        "p (centroid)": p_centroid,
    }
pd.DataFrame(rows).T.round(4)

,std normy,rho (norma),p (norma),rho (centroid),p (centroid)
Word2Vec (formy),0.3743,-0.7971,0.0000,0.8336,0.0
mmlw-roberta-base,0.2499,-0.0365,0.0099,-0.3397,0.0


## 6. Odpowiedzi na pytania pomocnicze

Liczby poniżej pochodzą z próbki 5000 zdań (`SEED = 42`).

### 6.1 Czy są zdania, które model uznał za podobne, ale różnią się znaczeniem?

Tak, w obu modelach (sekcja 5.1). Najwyraźniejszy przykład to `mmlw-roberta-base`: „POLECAMY" (`plus_m`) i „ODRADZAM!!!!" (`minus_m`) mają cosinus 0,98. Podobnie „Hotel nowy i zadbany." (`plus_m`) i „Hotel nowy a raczej niedokonczony." (`minus_m`) mają cosinus 0,97. Enkoder łączy zdania o tym samym temacie i podobnej formie, a przeciwny wydźwięk (negacja, antonim) wpływa na wektor słabo.

W Word2Vec najwyższe podobieństwa dotyczą głównie par, w których zdania różnią się tylko interpunkcją (np. „Polecam!!!" i „Polecam....", cosinus 1,00, etykiety `plus_m` i `minus_m`). Znaki interpunkcyjne nie wchodzą do słownika modelu, więc wektor zależy od samych słów. Jednocześnie pokazuje to ograniczenie danych: zdanie wyrwane z kontekstu recenzji nie niesie pełnej informacji o wydźwięku.

### 6.2 Czy są zdania semantycznie podobne, ale uznane za odległe, bo użyto innych słów?

Dowody są umiarkowane. Pary o tej samej etykiecie i najniższym cosinusie (sekcja 5.2) w `mmlw-roberta-base` to długie zdania `zero` o zupełnie różnych tematach (nauka, hotel, medycyna), więc niski cosinus jest tam poprawny, a nie błędny. W Word2Vec najniższe cosinusy dotyczą bardzo krótkich zdań („Była.", „Bezczelna?"), których wektor opiera się na jednym słowie.

Mocniejszy dowód daje zestawienie par, które `mmlw-roberta-base` uznaje za bliskie (cosinus powyżej 0,93), a Word2Vec za odległe (cosinus poniżej 0,08) przy zerowym nakładaniu słów. Przykład to „RAZ JESZCZE!!!!" i „POLECAMY": oba zdania wyrażają rekomendację innymi słowami, a Word2Vec ma cosinus 0,01. Zastrzeżenie: wszystkie takie pary to 1–2-wyrazowe hasła, więc część różnicy wynika z braku pokrycia słownika przez Word2Vec, a nie tylko z doboru słów. Część par jest też fałszywie podobna wg mmlw (np. „Bezczelna?" i „POLECAMY", cosinus 0,96).

### 6.3 Czy długość tekstu wpływa na „wyrazistość" wektora?

Tak, szczególnie w modelu statycznym (sekcja 5.3):

| Model | rho (`n_tokens`, norma) | rho (`n_tokens`, podobieństwo do centroidu klasy) |
| --- | --- | --- |
| Word2Vec (formy) | -0,80 | +0,83 |
| mmlw-roberta-base | -0,04 | -0,34 |

W Word2Vec im dłuższe zdanie, tym mniejsza norma średniej wektorów, bo wektory słów o różnych kierunkach się znoszą. Długie zdania leżą też bliżej centroidu klasy, ale to nie znaczy, że są lepiej reprezentowane. Uśrednianie ściąga wszystkie długie zdania w stronę wspólnego „środka" przestrzeni, więc tracą wyrazistość. Krótkie zdania zależą od pojedynczych słów, więc ich wektory są bardziej zmienne.

W `mmlw-roberta-base` norma prawie nie zależy od długości (rho -0,04), ale długie zdania są nieco dalej od centroidu swojej klasy (rho -0,34). Prawdopodobnie dlatego, że zawierają więcej różnorodnych treści. Nie sprawdzaliśmy bezpośrednio par zdań o tym samym sensie i różnej długości, więc wniosek o nich opiera się na tych korelacjach.

### 6.4 Który model lepiej separuje klasy z anotacji i dlaczego?

Nieznacznie lepszy jest `mmlw-roberta-base` (silhouette 0,013 wobec -0,038 dla Word2Vec), ale obie wartości są bliskie zeru, więc żaden model nie rozdziela dobrze czterech klas wydźwięku (`plus_m`, `minus_m`, `zero`, `amb`). Na wykresach t-SNE widać tylko wysepkę `zero` i słabszą wysepkę `minus_m`, reszta jest wymieszana.

Wynika to z natury zadania: oba modele osadzają przede wszystkim temat i budowę zdania, a nie wydźwięk. Zdania z pozytywnych i negatywnych opinii o hotelu są tematycznie bliższe sobie niż zdania o hotelu i o lekarzu (patrz sekcja 5.1). Dodatkowo etykieta pochodzi od całej recenzji, więc pojedyncze zdanie może jej nie nieść. Do rozdzielenia klas potrzebny byłby klasyfikator na tych osadzeniach lub dostrajanie modelu (Zadanie 2), a nie sam układ przestrzeni.